## Заполнение `probe_results.csv` (получение зондов)

## Зонд 1: Влияние длины ответа

Проверяем смещение выбора в пользу более длинного ответа.

Смотрим, какой вердикт дала каждая из моделей (GPT-4 и Claude) и как часто модель выбрала более длинный ответ.

Формула:

$
\text{longer\_response\_rate} =
\frac{n_{\text{longer}}}{n_{\text{objects}}}
$

где:

- $n_{\text{longer}}$ — число случаев, когда модель выбрала более длинный ответ;
- $n_{\text{objects}}$ — число проверенных объектов с неравной длиной ответов.

Объекты с равной длиной исключены из подсчета, потому что для них правило «длиннее — лучше» не даeт однозначного вердикта.

In [1]:
import os
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

slice_df = pd.read_json("audit/slice.jsonl", lines=True)
scorer_df = pd.read_json("audit/scorer.jsonl", lines=True)

gpt4 = scorer_df[scorer_df["scorer"] == "gpt4"].set_index("id")["verdict"]
claude = scorer_df[scorer_df["scorer"] == "claude"].set_index("id")["verdict"]

merged = slice_df.copy()
merged["verdict_gpt4"] = merged["id"].map(gpt4)
merged["verdict_claude"] = merged["id"].map(claude)

merged["len_1"] = merged["output_1"].str.len()
merged["len_2"] = merged["output_2"].str.len()

# Какой ответ длиннее: A (output_1) или B (output_2), None при равной длине
def _longer(row):
    if row["len_1"] > row["len_2"]:
        return "A"
    if row["len_2"] > row["len_1"]:
        return "B"
    return None

merged["longer"] = merged.apply(_longer, axis=1)

# Исключаем объекты с равной длиной
unequal = merged[merged["longer"].notna()].copy()
n_objects = len(unequal)

# Выбрал ли оценщик более длинный ответ 
unequal["gpt4_chose_longer"] = unequal["verdict_gpt4"] == unequal["longer"]
unequal["claude_chose_longer"] = unequal["verdict_claude"] == unequal["longer"]

gpt4_rate = unequal["gpt4_chose_longer"].mean()
claude_rate = unequal["claude_chose_longer"].mean()

print(f"Объектов с неравной длиной: {n_objects}")
print(f"GPT-4 выбрал более длинный ответ: {gpt4_rate:.3f}")
print(f"Claude выбрал более длинный ответ: {claude_rate:.3f}")

# Записываем зонд в probe_results.csv 
probe_results = pd.DataFrame([
    {"probe": "longer_response_rate_gpt4", "n": n_objects, "metric": "share",
     "value": round(gpt4_rate, 4),
     "interpretation": "Доля случаев, когда GPT-4 выбрал более длинный ответ среди объектов с различной длиной ответов."},
    {"probe": "longer_response_rate_claude", "n": n_objects, "metric": "share",
     "value": round(claude_rate, 4),
     "interpretation": "Доля случаев, когда Claude выбрал более длинный ответ среди объектов с различной длиной ответов."},
])

path = os.path.abspath("audit/probe_results.csv")
existing = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(
    columns=["probe", "n", "metric", "value", "interpretation"])
probe_results = pd.concat([existing, probe_results], ignore_index=True)
probe_results.to_csv(path, index=False, encoding="utf-8")

print(f"Обновлён файл: {path}")


Объектов с неравной длиной: 47
GPT-4 выбрал более длинный ответ: 0.660
Claude выбрал более длинный ответ: 0.660
Обновлён файл: c:\My Documents\Магистратура\Семестр 3\Нейроные сети (RL)\Практики\Практика 3\audit\probe_results.csv


## Зонд 2: Позиционное смещение (order bias)

Проверяем позиционное смещение выбора оценщиков, т.е. выбор первого показанного ответа.

Проверяем, как часто GPT-4 и Claude выбирают именно первый показанный ответ.

Формула:

$
\text{first\_shown\_rate} =
\frac{n_{\text{first}}}{n_{\text{objects}}}
$

где:

- $n_{\text{first}}$ — число случаев, когда модель выбрала первый показанный ответ;
- $n_{\text{objects}}$ — число проверенных объектов.

Если значение заметно выше 0.5, это возможный позиционный bias: модель может предпочитать не более качественный ответ, а тот, который был показан первым.

In [2]:
# Считаем доли выбора первого варианта
# Первый показанный ответ - всегда output_1, то есть вердикт A
merged["gpt4_chose_first"] = merged["verdict_gpt4"] == "A"
merged["claude_chose_first"] = merged["verdict_claude"] == "A"

n = len(merged)
gpt4_rate = merged["gpt4_chose_first"].mean()
claude_rate = merged["claude_chose_first"].mean()

print(f"Объектов: {n}")
print(f"GPT-4 выбрал первый показанный ответ: {gpt4_rate:.3f}")
print(f"Claude выбрал первый показанный ответ: {claude_rate:.3f}")

# Записываем зонд в probe_results.csv
probe_results = pd.DataFrame([
    {"probe": "first_shown_rate_gpt4", "n": n, "metric": "share",
     "value": round(gpt4_rate, 3),
     "interpretation": "Доля случаев, когда GPT-4 выбрал первый показанный ответ (output_1). Значение '< 0.5' - смещение в пользу второго ответа, '= 0.5' - нет смещения, '> 0.5' - смещение в пользу первого ответа."},
    {"probe": "first_shown_rate_claude", "n": n, "metric": "share",
     "value": round(claude_rate, 3),
     "interpretation": "Доля случаев, когда Claude выбрал первый показанный ответ (output_1). Значение '< 0.5' - смещение в пользу второго ответа, '= 0.5' - нет смещения, '> 0.5' - смещение в пользу первого ответа."},
])

path = os.path.abspath("audit/probe_results.csv")
existing = pd.read_csv(path)
probe_results = pd.concat([existing, probe_results], ignore_index=True)
probe_results.to_csv(path, index=False, encoding="utf-8")

print(f"Обновлён файл: {path}")


Объектов: 52
GPT-4 выбрал первый показанный ответ: 0.462
Claude выбрал первый показанный ответ: 0.423
Обновлён файл: c:\My Documents\Магистратура\Семестр 3\Нейроные сети (RL)\Практики\Практика 3\audit\probe_results.csv
